# 00 — Data Sanity Checks

**Purpose:** verify that the raw data pipeline in `src/data.py` is correct *before* any training happens.

**Why this notebook exists:** roughly 80% of "the model isn't learning" bugs trace back to the data layer, a chromosome-name mismatch, an off-by-one in window extraction, or a leaky train/test split. Those bugs are silent: the model trains, the loss goes down, and the AUROC looks plausible, but the model has learned the wrong thing. This notebook is the checklist that catches them.

**Prerequisites:**
- `data/raw/creb1_peaks.bed` (ENCODE narrowPeak, GRCh38)
- `data/raw/Homo_sapiens.GRCh38.dna.primary_assembly.fa` (Ensembl GRCh38)

If either is missing, run `bash scripts/download_data.sh` from the repo root first.

**When to run:** after any change to `src/data.py`, and once on a fresh clone.

In [ ]:
from src.notebook_setup import set_repo_root
set_repo_root()

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from pyfaidx import Fasta

from src.data import (
    CANONICAL_CHROMS, DataConfig, build_dataset,
    load_peaks, one_hot, extract_window, _peak_center,
)

## 1. The BED file looks sane

We expect a narrowPeak file with 10 columns, coordinates in GRCh38, and chromosome names like `chr1` (which `load_peaks` normalizes to `1`).

**What to verify in the output below:**
- `chrom` column is `object` (string), not `int64`
- Only canonical chromosomes appear (`1`–`22`, `X`, `Y`) — no `chrUn_*`, no alt contigs, no `MT`
- `summit >= peak length` is **0** (a nonzero value means the summit falls outside the peak interval, which is malformed narrowPeak)

In [ ]:
cfg = DataConfig()
peaks = load_peaks(cfg.bed_path)

print(f"peaks: {len(peaks):,}")
print(f"columns: {list(peaks.columns)}")
print(f"chrom dtype: {peaks.chrom.dtype}")
print(f"start dtype: {peaks.start.dtype}")
display(peaks.head())
print("\nper-chromosome counts:")
display(peaks.chrom.value_counts().sort_index())

print(f"\nsummit == -1 rows: {(peaks.summit == -1).sum():,}")
print(f"summit >= peak length (bad): {(peaks.summit >= (peaks.end - peaks.start)).sum():,}")

## 2. The FASTA and BED agree on chromosome names

This is the check that catches the single most common data bug in this pipeline: the BED uses `chr1` and the FASTA uses `1` (or vice versa). `load_peaks` strips `chr`, but if the FASTA headers have a different convention (e.g. `chr1` vs `1` vs `NC_000001.11`), every `extract_window` call will `KeyError`, or worse, silently return empty sequences.

**What to verify below:**
- `in BED but not FASTA` must be **empty**
- `chr1 length` should be ~248,956,422 bp for GRCh38 (if it's ~3M, you downloaded the wrong file)
- The first 60 bp of chr1 should start with `N`s — GRCh38 chr1 begins with ~10,000 `N`s (telomeric repeat). If it starts with clean `ACGT`, you're looking at the wrong assembly or a masked version.

In [ ]:
fa = Fasta(str(cfg.fasta_path), as_raw=True)

fa_keys = set(fa.keys())
bed_keys = set(peaks.chrom.unique())

print(f"FASTA sequences: {len(fa_keys):,}")
print(f"BED chroms after filter: {sorted(bed_keys)}")
print(f"in BED but not FASTA: {bed_keys - fa_keys}")
print(f"canonical chroms missing from FASTA: {[c for c in CANONICAL_CHROMS if c not in fa_keys]}")

print(f"\nchr1 length: {len(fa['1']):,} bp")
print(f"first 60 bp of chr1: {str(fa['1'][1:61]).upper()}")

## 3. Window extraction produces the right bases

`extract_window` is where the BED↔pyfaidx 0-based/1-based conversion happens. An off-by-one here corrupts every positive sample and you won't notice until the model underperforms. This cell prints a real window so you can eyeball it.

**What to verify below:**
- `bases covered` should be **500 / 500** for an interior peak (no clipping). If less, the peak is near a chromosome edge.
- The reconstructed sequence should look like DNA, if it's mostly `N`s, either the peak is in a centromere/telomere, or the offset is wrong.
- The sequence should correspond to what you'd see if you looked up the same coordinates in UCSC Genome Browser (spot-check one if you're paranoid).

In [ ]:
row = peaks.iloc[0]
center = _peak_center(row)
x = extract_window(fa, row.chrom, center, cfg.window)

print(f"peak: {row.chrom}:{row.start}-{row.end}  summit={row.summit}  center={center}")
print(f"window shape: {x.shape}   sum per base: {x.sum(axis=0)[:5]} ...")
print(f"bases covered (nonzero columns): {(x.sum(axis=0) > 0).sum()} / {cfg.window}")

seq = "".join("ACGT"[int(np.argmax(x[:, i]))] if x[:, i].any() else "N"
              for i in range(x.shape[1]))
print(f"\nreconstructed (first 80): {seq[:80]}")
print(f"N count in window: {seq.count('N')}")

## 4. Build the full dataset

This runs the whole pipeline (positives + negatives) and caches the result to `data/processed/dataset_w{window}_n{neg_per_pos}_s{seed}.npz`. On the first run it takes a few minutes (the genome is 3 GB); after that it loads from cache in under a second.

**What to verify below:**
- `X_pos` and `X_neg` both have shape `(N, 4, 500)` and dtype `float32`
- `chroms_pos` and `chroms_neg` are `object` (arrays of strings), shape `(N,)`

If `chroms_*` are missing, you have a stale cache from an earlier version of `src/data.py`, delete `data/processed/*.npz` and re-run.

In [ ]:
data = build_dataset(cfg)

for k, v in data.items():
    print(f"{k}: shape={v.shape}  dtype={v.dtype}")

## 5. Class balance and chromosome distribution

With `neg_per_pos=1`, positives and negatives should be exactly balanced. More subtly, the **per-chromosome distribution** of negatives should roughly track positives, otherwise the model can exploit chromosome length as a shortcut ("if this window is from chr1, predict positive").

`sample_negatives` weights chromosome sampling by length, so the `diff` column below should be small.

**What to verify below:**
- ratio ≈ 1.00
- `max |pos − neg| chrom fraction` < ~0.05 (a small gap is expected because peaks cluster on gene-rich chromosomes)
- If `neg_dist` looks *uniform* across all chromosomes, the length-weighting in `sample_negatives` is broken. chr1 should be ~8% of negatives, chrY ~2%.

In [ ]:
print(f"positives: {len(data['X_pos']):,}")
print(f"negatives: {len(data['X_neg']):,}")
print(f"ratio: 1 : {len(data['X_neg']) / len(data['X_pos']):.2f}")

pos_dist = pd.Series(data['chroms_pos']).value_counts(normalize=True).sort_index()
neg_dist = pd.Series(data['chroms_neg']).value_counts(normalize=True).sort_index()

dist = pd.DataFrame({"pos_frac": pos_dist, "neg_frac": neg_dist})
dist["diff"] = dist["pos_frac"] - dist["neg_frac"]
display(dist.round(4))
print(f"\nmax |pos - neg| chrom fraction: {dist['diff'].abs().max():.4f}")

## 6. The train/test split is leak-free

This is the **most important check in the notebook**. If a chromosome appears in both train and test, near-identical sequences leak across the boundary and your test AUROC is meaningless (a peak at chr1:1,000,000 and its 50-bp-shifted neighbor at chr1:1,000,050 are effectively duplicates).

`split_by_chromosome` holds out entire chromosomes, so the overlap between train and test chromosomes should be **empty**.

**What to verify below:**
- `overlap` is the empty set
- `test chroms` is exactly `{'1', '2'}` (matching `cfg.test_chroms`)
- train/test positive rates are both ≈ 0.5 (because the 1:1 ratio holds in both splits)

In [ ]:
from src.data import split_by_chromosome

split = split_by_chromosome(data, test_chroms=cfg.test_chroms)

print(f"train: {len(split['y_train']):,}  (positives: {int(split['y_train'].sum()):,})")
print(f"test:  {len(split['y_test']):,}  (positives: {int(split['y_test'].sum()):,})")

train_chroms = set(split['chroms_train'])
test_chroms_set = set(split['chroms_test'])

print(f"\ntrain chroms: {sorted(train_chroms)}")
print(f"test chroms:  {sorted(test_chroms_set)}")
print(f"overlap:      {train_chroms & test_chroms_set}")

print(f"\ntrain positive rate: {split['y_train'].mean():.3f}")
print(f"test  positive rate: {split['y_test'].mean():.3f}")

## Done

If every check above passed, `src/data.py` is trustworthy and you can move on to `01_train.ipynb`.

If any check failed, **do not train** the model will learn the wrong thing and the evaluation numbers will be meaningless. Fix the data bug first, then re-run this notebook.